# Outage-Based Outer Budget Allocation

**목적**: ζ bin별 outage cliff 위치가 다름을 이용하여, easy channel은 budget 줄이고 hard channel은 budget 늘려서 평균 budget 동일하게 유지하면서 outage probability를 줄이는 실험.

**이전 시도 (mean NMSE)**: distortion 곡선이 bin간 평행 → equal allocation이 최적 → 개선 없음  
**이번 시도 (outage)**: outage는 threshold 기반 cliff → cliff 위치 다르면 unequal allocation 유리

---
### 실행 순서
1. **Cell 1-3**: 환경 설정 (Colab GPU)
2. **Cell 4**: Step 1 — per-bin outage curves 생성 (GPU, ~30-60분)
3. **Cell 5**: Step 2 — budget allocation 최적화 (CPU, ~1분)
4. **Cell 6-7**: 시각화 및 분석

## Cell 1: Drive Mount & Path Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/MambaCompression/MambaIC'
os.chdir(PROJECT_ROOT)
print(f'Working dir: {os.getcwd()}')
print(f'Data exists: {os.path.exists("data/DATA_Htestout.mat")}')
print(f'Model exists: {os.path.exists("saved_models/mamba_transnet_L2_dim512_baseline/best.pth")}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/MyDrive/MambaCompression/MambaIC
Data exists: True
Model exists: True


## Cell 2: Dependencies (VMamba CUDA build if needed)

In [2]:
# Run setup_colab.py if VMamba/mamba-ssm not installed
try:
    import mamba_ssm
    print(f'mamba_ssm already installed: {mamba_ssm.__version__}')
except ImportError:
    print('Installing mamba-ssm + VMamba CUDA kernels...')
    %run ../setup_colab.py

Installing mamba-ssm + VMamba CUDA kernels...
=== 1. Core Dependencies ===
[  0.0s] pip install core deps...
[  5.4s] core deps done

=== 2. VMamba CUDA Kernel (ss2d) ===
Current GPU: Tesla T4 (sm_75)
Cache arch matches current GPU (sm_75) ✓
Cache found! Restoring 1 kernel files...
[  8.3s] copying .so from cache...
  Restored: selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so -> /usr/local/lib/python3.12/dist-packages/selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so
[  8.6s] .so copy done
[  8.6s] importing selective_scan_cuda_oflex...
[  8.6s] selective_scan_cuda_oflex imported OK (sm_75)
selective_scan_cuda_oflex imported OK (sm_75)
[  8.6s] setup_colab.py done

=== Setup Complete ===
Project: /content/drive/MyDrive/MambaCompression


## Cell 3: Verify Prerequisites

In [3]:
import torch
import pandas as pd
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Check required CSVs
csv_dir = 'results/csv'
required = [
    'segment_dp_omegas.csv',
    'rpmpq_v2_zeta.csv',
    'rpmpq_v2_perfect_rates.csv',
]
# kappa CSV (either name)
kappa_ok = os.path.exists(f'{csv_dir}/rpmpq_v2_kappa.csv') or os.path.exists(f'{csv_dir}/rpmpq_v2_step1_nmse_kappa.csv')

all_ok = True
for f in required:
    exists = os.path.exists(f'{csv_dir}/{f}')
    status = 'OK' if exists else 'MISSING'
    print(f'  {f}: {status}')
    if not exists:
        all_ok = False
print(f'  kappa CSV: {"OK" if kappa_ok else "MISSING"}')
if not kappa_ok:
    all_ok = False

if all_ok:
    print('\n All prerequisites satisfied. Ready to run.')
else:
    print('\n MISSING files. Run rpmpq_v2.py and segment_dp_policy.py first.')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
  segment_dp_omegas.csv: OK
  rpmpq_v2_zeta.csv: OK
  rpmpq_v2_perfect_rates.csv: OK
  kappa CSV: OK

 All prerequisites satisfied. Ready to run.


## Cell 4: Step 1 — Build Per-Bin Outage Curves (GPU)

K=5 bins × 25 saving levels = 125 (bin, saving) pairs.  
각 pair에서 segment DP + inference + outage 계산.  
**예상 시간: 30-60분** (GPU에 따라 다름). 중간 저장되므로 중단 후 재실행 가능.

In [6]:
import sys
sys.path.insert(0, PROJECT_ROOT)

from analysis.budget_allocation_outage import run_step1_curves

df_curves = run_step1_curves(objective='nmse')

  STEP 1: Building per-bin outage-vs-budget curves
  Device: CUDA
[INFO] Building: UE Encoder [mamba-L2] + BS Decoder [transnet-L2]
  Test samples: 20000
    Bin 0: 4000 samples
    Bin 1: 4000 samples
    Bin 2: 4000 samples
    Bin 3: 4000 samples
    Bin 4: 4000 samples

  Loading cached segment omegas...

  DP objective for policy selection: nmse
  Budget savings range: 85.0% -- 97.00000000000017%
  Gammas: [0.99, 0.98, 0.95]

  Total (bin x saving) combinations: 305
  Already cached: 0,  remaining: 305


Outage curves: 100%|██████████| 305/305 [04:33<00:00,  1.11it/s]

  Saved outage curves -> /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/outage_curves_per_bin.csv

  Outage curve summary:

    gamma = 0.99
      Bin 0: outage [0.2585, 1.0000]  cliff near 86.8% saving
      Bin 1: outage [0.3392, 1.0000]  cliff near 86.8% saving
      Bin 2: outage [0.4143, 1.0000]  cliff near 86.8% saving
      Bin 3: outage [0.5962, 1.0000]  cliff near 86.8% saving
      Bin 4: outage [0.8420, 1.0000]  cliff near 86.8% saving

    gamma = 0.98
      Bin 0: outage [0.0143, 1.0000]  cliff near 87.6% saving
      Bin 1: outage [0.0203, 1.0000]  cliff near 87.6% saving
      Bin 2: outage [0.0325, 1.0000]  cliff near 87.4% saving
      Bin 3: outage [0.0465, 1.0000]  cliff near 87.2% saving
      Bin 4: outage [0.1452, 1.0000]  cliff near 87.0% saving

    gamma = 0.95
      Bin 0: outage [0.0000, 1.0000]  cliff near 89.2% saving
      Bin 1: outage [0.0000, 1.0000]  cliff near 89.0% saving
      Bin 2: outage [0.0000, 1.0000]  cliff near 89.0% saving
    

## Cell 5: Step 2 — Optimize Per-Bin Budget Allocation (CPU)

Greedy + Grid search로 population-weighted outage 최소화하는 per-bin budget 찾기.  
**예상 시간: ~1분.**

In [8]:
from analysis.budget_allocation_outage import run_step2_optimize

df_alloc = run_step2_optimize(df_curves)


  STEP 2: Optimising per-bin budget allocation (outage objective)

  --- gamma = 0.99 ---
    Population weights: ['0.200', '0.200', '0.200', '0.200', '0.200']
    Savings levels: 61
     85.2%: equal=0.5068  opt=0.4989  D=0.0079 (1.6%)  [greedy]  b0=85.6%  b1=85.4%  b2=85.0%  b3=85.0%  b4=85.0%
     85.3%: equal=0.5068  opt=0.4989  D=0.0079 (1.6%)  [greedy]  b0=85.6%  b1=85.4%  b2=85.0%  b3=85.0%  b4=85.0%
     85.6%: equal=0.5246  opt=0.5155  D=0.0091 (1.7%)  [greedy]  b0=85.8%  b1=85.8%  b2=85.4%  b3=85.4%  b4=85.6%
     85.7%: equal=0.5246  opt=0.5155  D=0.0091 (1.7%)  [greedy]  b0=85.8%  b1=85.8%  b2=85.4%  b3=85.4%  b4=85.6%
     86.0%: equal=0.5403  opt=0.5321  D=0.0083 (1.5%)  [greedy]  b0=86.2%  b1=85.8%  b2=85.8%  b3=85.8%  b4=86.4%
     86.1%: equal=0.5403  opt=0.5321  D=0.0083 (1.5%)  [greedy]  b0=86.2%  b1=85.8%  b2=85.8%  b3=85.8%  b4=86.4%
     86.6%: equal=0.6036  opt=0.5767  D=0.0269 (4.5%)  [greedy]  b0=86.8%  b1=86.8%  b2=86.4%  b3=86.2%  b4=86.8%
     86.7%: equal=

## Cell 6: Visualize Outage Curves Per Bin

핵심 확인: **bin별 cliff 위치가 다른지?** 다르면 adaptive allocation이 유효.

In [ ]:
import matplotlib.pyplot as plt

df_c = pd.read_csv('results/csv/outage_curves_per_bin.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 5))

for ax_idx, gamma in enumerate([0.99, 0.98, 0.95]):
    ax = axes[ax_idx]
    sub = df_c[df_c['gamma'] == gamma]
    
    for j in range(5):
        bsub = sub[sub['bin'] == j].sort_values('target_saving')
        label = f'Bin {j} ({"easy" if j == 0 else "hard" if j == 4 else ""}'
        label += f', n={int(bsub["n_samples"].iloc[0])})'
        ax.plot(bsub['target_saving'], bsub['outage'],
                'o-', color=colors[j], label=label, markersize=3)
    
    ax.set_xlabel('BOPs Saving (%)')
    ax.set_ylabel('Outage Probability')
    ax.set_title(f'γ = {gamma}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('Per-Bin Outage vs BOPs Saving — Cliff Position Matters!', fontsize=13)
plt.tight_layout()
plt.savefig('results/plots/outage_curves_per_bin.png', dpi=150, bbox_inches='tight')
plt.show()

# Print cliff positions
print('\nEstimated cliff positions (largest single-step outage jump):')
for gamma in [0.99, 0.98, 0.95]:
    print(f'  gamma={gamma}:')
    sub = df_c[df_c['gamma'] == gamma]
    for j in range(5):
        bsub = sub[sub['bin'] == j].sort_values('target_saving')
        outages = bsub['outage'].values
        savings = bsub['target_saving'].values
        diffs = np.diff(outages)
        if len(diffs) > 0:
            cliff_idx = np.argmax(diffs)
            print(f'    Bin {j}: cliff at ~{savings[cliff_idx]:.1f}% saving '
                  f'(outage jump: {diffs[cliff_idx]:.3f})')

## Cell 7: Allocation Results & Improvement Analysis

In [ ]:
df_a = pd.read_csv('results/csv/budget_allocation_outage.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, gamma in enumerate([0.99, 0.98, 0.95]):
    ax = axes[ax_idx]
    sub = df_a[df_a['gamma'] == gamma].sort_values('target_saving')
    
    ax.plot(sub['target_saving'], sub['equal_outage'],
            'k--', label='Equal allocation', linewidth=2)
    ax.plot(sub['target_saving'], sub['optimal_outage'],
            'r-', label='Optimal allocation', linewidth=2)
    
    # Shade improvement region
    ax.fill_between(sub['target_saving'],
                    sub['optimal_outage'], sub['equal_outage'],
                    where=sub['improvement'] > 0,
                    alpha=0.3, color='green', label='Improvement')
    
    ax.set_xlabel('Average BOPs Saving (%)')
    ax.set_ylabel('Population-Weighted Outage')
    ax.set_title(f'γ = {gamma}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Outage-Based Budget Allocation: Equal vs Optimal', fontsize=13)
plt.tight_layout()
plt.savefig('results/plots/budget_allocation_outage.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('\n=== Best improvements per gamma ===')
for gamma in [0.99, 0.98, 0.95]:
    sub = df_a[df_a['gamma'] == gamma]
    sub_pos = sub[sub['improvement'] > 1e-5]
    if len(sub_pos) > 0:
        best = sub_pos.loc[sub_pos['improvement_pct'].idxmax()]
        print(f'\n  gamma={gamma}: best improvement at {best["target_saving"]:.1f}% saving')
        print(f'    Equal outage:   {best["equal_outage"]:.4f}')
        print(f'    Optimal outage: {best["optimal_outage"]:.4f}')
        print(f'    Improvement:    {best["improvement"]:.4f} ({best["improvement_pct"]:.1f}%)')
        per_bin = [f'B{j}={best[f"saving_{j}"]:.1f}%' for j in range(5)]
        print(f'    Per-bin saving: {"  ".join(per_bin)}')
    else:
        print(f'\n  gamma={gamma}: No improvement found (curves may be parallel for outage too)')

# Show per-bin allocations at key saving levels
print('\n=== Per-bin budget allocation at key savings ===')
for gamma in [0.99]:
    sub = df_a[df_a['gamma'] == gamma]
    for s in [87.5, 90.0, 92.5, 95.0]:
        row = sub[sub['target_saving'].between(s-0.1, s+0.1)]
        if len(row) > 0:
            r = row.iloc[0]
            bins = [f'{r[f"saving_{j}"]:.1f}%' for j in range(5)]
            print(f'  {s:.1f}%: equal_out={r["equal_outage"]:.4f}  '
                  f'opt_out={r["optimal_outage"]:.4f}  '
                  f'D={r["improvement"]:.4f}  '
                  f'bins=[{", ".join(bins)}]')

## Cell 8: Diagnostic — 결과가 안 좋으면 확인할 것

만약 improvement ≈ 0이면, 아래를 확인:
1. **Cliff 위치가 bin간에 같은가?** → Cell 6의 cliff position 확인
2. **Outage 곡선의 shape이 같은가?** → 곡선이 평행이면 mean NMSE와 같은 문제
3. **K=5가 너무 coarse?** → K=10으로 올려서 재시도

In [ ]:
# Diagnostic: cliff separation analysis
df_c = pd.read_csv('results/csv/outage_curves_per_bin.csv')

print('=== Cliff Separation Analysis ===')
print('Question: Do different bins have cliffs at different savings?\n')

for gamma in [0.99, 0.98, 0.95]:
    cliffs = []
    sub = df_c[df_c['gamma'] == gamma]
    for j in range(5):
        bsub = sub[sub['bin'] == j].sort_values('target_saving')
        outages = bsub['outage'].values
        savings = bsub['target_saving'].values
        diffs = np.diff(outages)
        if len(diffs) > 0:
            cliff_idx = np.argmax(diffs)
            cliffs.append(savings[cliff_idx])
        else:
            cliffs.append(np.nan)
    
    spread = max(cliffs) - min(cliffs) if all(~np.isnan(c) for c in cliffs) else 0
    print(f'  gamma={gamma}: cliffs at {["{:.1f}".format(c) for c in cliffs]}')
    print(f'    Cliff spread: {spread:.1f}% saving')
    if spread >= 1.0:
        print(f'    >>> GOOD: {spread:.1f}% spread — adaptive allocation should help')
    else:
        print(f'    >>> WARNING: Only {spread:.1f}% spread — may not be enough')
    print()